

**Тема: Обучение модели YOLO на кастомном датасете и исследование влияния гиперпараметров на качество детекции**

**Цель работы:** Познакомиться с архитектурой YOLO на примере проверки гипотезы о релевантной метрике.

**Задачи:**

- Ознакомиться с архитектурой YOLO.
- Изучить метрики для анализа производительности модели, выбрать целевую метрику в соответствии с вариантом.
- Выбрать предметную область, сформировать гипотезу для проведения исследования.
- Собрать и проаннотировать данные, сформировать датасет.
- Провести fine-tuning предобученной модели YOLOv11 Nano/Small.
- Визуализировать и проанализировать результаты.
- На основе анализа сделать корректировку гиперпараметров/данных и провести вторую итерацию для повышения показателей.

### 1. Подготовка к обучению

#### 1.1 Метрики

Вариант 1 - Precision

Вариант 2 - Recall

Вариант 3 - F1

Вариант 4 - mAP@0.5

Вариант 5 - mAP@0.75


#### 1.2 Гипотеза

Гипотеза должна отражать характер уклона исследования с обоснованием, отталкиваясь от предметной области. *Одна лишь констатация необходимости достижения высокого значения целевой метрики не является обоснованием*.

#### 1.3 Данные

Соберите не менее 500 изображений из открытых источников. Можно пользоваться готовыми наборами данных, но важно проверить качество: разрешение изображений, качество аннотаций, баланс классов. При самостоятельном сборе данных можете воспользоваться терминальной утилитой ffmpeg для нарезки видео на кадры и любым удобным инструментом аннотирования (Roboflow, CVAT и тд).

#### 1.4 Предобработка

Примените методы аугментации к данным для расширения объема датасета для получения 1.5-2к изображений. Подготовьте данные к требуемому формату для обучающего процесса.

### 2. Обучение модели

#### 2.1 Подготовка окружения

Установите зависимости и библиотеки:

In [2]:
# импорт пакетов
import sys
!{sys.executable} -m pip install -q ultralytics roboflow

import os
import glob
import random
import numpy as np
import torch
from ultralytics import YOLO
from roboflow import Roboflow
from IPython.display import Image, display

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.8/95.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 69.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Torch: 2.10.0+cu128
CUDA available: True
Using device: cuda


#### 2.2 Подготовка модели

Загрузите предобученную модель, определите устройство, переведите модель в режим инференса. Не используйте размер модели больше чем Small для достижения лучших показателей на стандартных гиперпараметрах (особенно imgsz)

In [3]:
model = YOLO("yolov8n.pt")
model.to(device)
print("Модель загружена")

Модель загружена


#### 2.3 Загрузка и предобработка изображений


Затем импортируйте датасет в проект и выполните трансформацию данных (при использовании Roboflow трансформация выполняется на этапе предобработки):

In [ ]:
# загрузка датасета
rf = Roboflow(api_key="fP9RU4GRTrlWgwOSVlXI")
project = rf.workspace("grizzlyichs-workspace").project("vehicles-rasr0-nj9sa")
version = project.version(1)
dataset = version.download("yolov8")

data_yaml_path = f"{dataset.location}/data.yaml"
print("Dataset yaml:", data_yaml_path)

# Быстрая проверка структуры
import glob
train_imgs = glob.glob(f"{dataset.location}/train/images/*")
valid_imgs = glob.glob(f"{dataset.location}/valid/images/*")
test_imgs  = glob.glob(f"{dataset.location}/test/images/*")
print("train images:", len(train_imgs))
print("valid images:", len(valid_imgs))
print("test images:", len(test_imgs))

loading Roboflow workspace...
loading Roboflow project...
Dataset yaml: C:\Users\podzo\vehicles-1/data.yaml
train images: 3500
valid images: 1000
test images: 500


#### 2.4 Обучение, оценка модели и визуализация результатов

Проведите обучение модели, проанализируйте кривые обучения, метрики и тестовые данные. Сделайте вывод и корректироваки для достижения лучших показателей

In [4]:
# обучение, оценка, визуализация

import os, glob, random
import torch
from ultralytics import YOLO
from IPython.display import Image, display


device = "cuda"
print("Using device:", device)


assert os.path.exists(data_yaml_path), f"Не найден файл: {data_yaml_path}"
dataset_root = os.path.dirname(data_yaml_path)

train_imgs = glob.glob(f"{dataset_root}/train/images/*")
valid_imgs = glob.glob(f"{dataset_root}/valid/images/*")
print("train images:", len(train_imgs))
print("valid images:", len(valid_imgs))


run1 = "yolov8n_fast_run1"
project_dir = "runs/detect"

model = YOLO("yolov8n.pt")

results1 = model.train(
    data=data_yaml_path,
    epochs=2,
    imgsz=512,
    batch=8,
    device=device,
    workers=0,
    optimizer="AdamW",
    lr0=0.003,
    project=project_dir,
    name=run1,
    plots=True
)


metrics1 = model.val(data=data_yaml_path, device=device)
print("\n=== METRICS RUN1 ===")
print(metrics1)


results_png = f"{project_dir}/{run1}/results.png"
if os.path.exists(results_png):
    display(Image(results_png))
else:
    print("Не найден results.png:", results_png)


sample = random.sample(valid_imgs, k=min(5, len(valid_imgs)))

pred_name = f"{run1}_pred"
model.predict(
    source=sample,
    conf=0.25,
    save=True,
    project=project_dir,
    name=pred_name
)

pred_imgs = sorted(glob.glob(f"{project_dir}/{pred_name}/*.jpg"))[:5]
for p in pred_imgs:
    display(Image(p))

print("\nBest weights:", f"{project_dir}/{run1}/weights/best.pt")
print("Last weights:", f"{project_dir}/{run1}/weights/last.pt")

Using device: cuda


NameError: name 'data_yaml_path' is not defined

#### 2.5 Вторая итерация

Проведите процедуры для достижения высоких показателей (корректировка данных/гиперпараметров), сделайте вывод


In [ ]:
# исследование
import os, glob, random
from ultralytics import YOLO
from IPython.display import Image, display

device = "cpu"
project_dir = "runs/detect"
run2 = "yolov8n_fast_run2_tuned"

model2 = YOLO("yolov8n.pt")

model2.train(
    data=data_yaml_path,
    epochs=4,
    imgsz=512,
    batch=8,
    device=device,
    workers=0,
    optimizer="AdamW",
    lr0=0.002,
    weight_decay=0.0005,
    cos_lr=True,
    augment=True,
    mosaic=1.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    project=project_dir,
    name=run2,
    plots=True
)

metrics2 = model2.val(data=data_yaml_path, device=device)
print("\n=== METRICS RUN2 ===")
print(metrics2)


results_png_2 = f"{project_dir}/{run2}/results.png"
if os.path.exists(results_png_2):
    display(Image(results_png_2))
else:
    print("Не найден results.png:", results_png_2)


dataset_root = os.path.dirname(data_yaml_path)
valid_imgs = glob.glob(f"{dataset_root}/valid/images/*")
sample = random.sample(valid_imgs, k=min(5, len(valid_imgs)))

pred2_name = f"{run2}_pred"
model2.predict(
    source=sample,
    conf=0.25,
    save=True,
    project=project_dir,
    name=pred2_name
)

pred_imgs_2 = sorted(glob.glob(f"{project_dir}/{pred2_name}/*.jpg"))[:5]
for p in pred_imgs_2:
    display(Image(p))

print("\nBest weights:", f"{project_dir}/{run2}/weights/best.pt")
print("Last weights:", f"{project_dir}/{run2}/weights/last.pt")

Ultralytics 8.4.19  Python-3.9.13 torch-2.8.0+cpu CPU (AMD Ryzen 7 5700X 8-Core Processor)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=C:\Users\podzo\vehicles-1/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=4, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=512, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8n_fast_run2_tuned, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=Tr